# 1. Measure slices for liblsqecc layouts

## Before running

Use a source checkout. From its root, prepare a Python environment and install:

```sh
python -m pip install -e '.[notebooks]'
python scripts/bootstrap.py --jobs 4
python scripts/build_liblsqecc_adapter.py --source .deps/liblsqecc --build .deps/liblsqecc/build --out .deps/adapter
jupyter lab notebooks
```

The native build requires Git, CMake, a C++20 compiler and liblsqecc development
libraries; see [the integration guide](../docs/integration.md). The adapter is our
executable around liblsqecc that accepts custom layouts. Build it once, then reuse
it. Choose the kernel belonging to the environment where you installed the package.
Use **Restart Kernel and Run All**. Each run creates a separate evidence directory.

These small examples teach integration, not algorithm performance. All compilation
uses **wave scheduling with local instruction decomposition enabled**. Area means
all cells in the rectangular grid; slices are completed compiler frames; their
product is measured in tile-slices, not physical qubit-cycles. `M` cells are abstract
magic-state supplies, with compiler production timing fixed at 10.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import json
import subprocess

# Start Jupyter from the repository root or its notebooks directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/lattice_floorplan").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository checkout.")

# Change these two paths if you built the adapter somewhere else.
ADAPTER = ROOT / ".deps/adapter/liblsqecc_adapter"
EXPORTER = ROOT / ".deps/adapter/export_layout"
for executable in (ADAPTER, EXPORTER):
    if not executable.is_file():
        raise FileNotFoundError(f"Build the adapter first; missing {executable}")

from lattice_floorplan.compiler import Compiler, CompilerConfig
from lattice_floorplan.geometry import GridState, GeometryOracle
from lattice_floorplan.layout import Layout

# This example is already synthesized into Clifford+T gates.
qasm = (ROOT / "docs/_static/example.qasm").read_text()
compiler = Compiler(ADAPTER, CompilerConfig(local=True, timeout=30))
(ROOT / "runs").mkdir(exist_ok=True)
RUN = Path(mkdtemp(prefix="notebook-", dir=ROOT / "runs"))
print("Evidence directory:", RUN.relative_to(ROOT))


## Generate a native grid and compile it

`export_layout` calls liblsqecc's built-in layout generators. We evaluate all three
names below on the same five-qubit example. The matching `preprocess_layout` is
essential: it preserves each generator's circuit preprocessing. Native compact
grids can fail local decomposition; failures are reported with no numerical score.
An exported native grid initially assigns logical identities in row-major order.


In [ ]:
native_results = []
for mode in ("edpc", "compact", "compact_no_clogging"):
    grid_text = subprocess.run(
        [str(EXPORTER), mode, "5"], check=True, capture_output=True, text=True
    ).stdout
    (RUN / f"{mode}.txt").write_text(grid_text)
    layout = Layout.parse(grid_text)
    record = compiler.run(qasm, layout, RUN / mode, input_format="qasm",
                          retain_slices=True, preprocess_layout=mode)
    area = len(layout.rows) * len(layout.rows[0])
    slices = record["slices"] if record["status"] == "success" else None
    native_results.append({"generator": mode, "status": record["status"],
                           "area_tiles": area, "slices": slices,
                           "volume": area * slices if slices is not None else None,
                           "reason": record.get("reason")})
print(json.dumps(native_results, indent=2))
assert next(r for r in native_results if r["generator"] == "edpc")["status"] == "success"


## Inspect the actual slices

Successful evidence directories contain `slices.json`, the full sequence of
compiler frames. `record["slices"]` is the validated duration to use; do not count
stdout lines or interpret a failed partial schedule as a result. To evaluate your
own circuit, replace `qasm` and the exporter qubit count together. It must already
use the supported synthesized gate set.


In [ ]:
frames_path = RUN / "edpc" / "slices.json"
frames = json.loads(frames_path.read_text())
print("EDPC completed slices:", native_results[0]["slices"])
print("Saved frame container type:", type(frames).__name__)
print("Grid:")
print((RUN / "edpc.txt").read_text())
